# Dataset Annotation IE

## Assignment Information

| Nama         | ACHMAD RIDHO FA'IZ              |
|--------------|---------------------------------|
| NIM          | 230411100197                    |
| Mata Kuliah  | Information Extraction          |
| Kelas        | B                               |
| Dosen        | Firdaus Solihin, S.Kom., M.Kom. |
| Universitas  | Universitas Trunodjoyo Madura   |
| Tanggal      | 18 October 2026                 |

Tugas ini membangun dataset berlabel *named entity recognition* dari 208
putusan pidana umum PN Ngawi hasil scraping tugas pertama. Setiap dokumen
dianotasi pada skema IOB untuk dua belas entitas: nomor perkara, terdakwa,
tindak pidana, pasal, tuntutan, putusan, tanggal, hakim ketua, hakim
anggota, panitera, penuntut umum, dan penasihat hukum. Keluaran disimpan
sebagai `doc, sentence, token, tag` pada `data/IE/dataset/`.

## A. Corpus Inventory

Korpus berasal dari 208 putusan pidana umum PN Ngawi yang dikumpulkan pada
25 September 2026 dari `putusan3.mahkamahagung.go.id`. Situs menolak akses
otomatis lewat Cloudflare (respons 403), sehingga berkas PDF diunduh secara
manual lalu diproses lokal. Teks PDF diekstrak dengan `pypdf` dan disimpan
sebagai *cache*; putusan yang berkasnya tidak tersedia dianotasi dari
metadata dan kolom `catatan_amar`.

In [1]:
import hashlib
import itertools
import re
import unicodedata
from pathlib import Path

import pandas as pd
from pypdf import PdfReader

DATA = Path("..", "..", "data", "IE")
META = DATA / "PN-Ngawi" / "IE-PN-Ngawi-230411100197.csv"
PDF_DIR = DATA / "dataset" / "PDF"
TXT_CACHE = DATA / "dataset" / "TXT"
OUT_CSV = DATA / "dataset" / "PN-NGW-230411100197.csv"
OUT_TXT = DATA / "dataset" / "PN-NGW-230411100197.txt"
TXT_CACHE.mkdir(parents=True, exist_ok=True)
print(f"metadata {META.name} | korpus {TXT_CACHE.parent} | keluaran {OUT_CSV.name}")

metadata IE-PN-Ngawi-230411100197.csv | korpus ../../data/IE/dataset | keluaran PN-NGW-230411100197.csv


Berkas metadata menyatukan seluruh hasil scraping: satu baris per putusan
dengan dua puluh satu kolom, termasuk nama terdakwa, hakim, panitera,
penuntut umum, nomor perkara, dan catatan amar.

In [2]:
meta = pd.read_csv(META, dtype=str).fillna("")
print(f"{len(meta)} putusan | {meta.shape[1]} kolom metadata | {meta['nomor'].nunique()} nomor perkara")

208 putusan | 21 kolom metadata | 208 nomor perkara


In [3]:
tipe = meta["nomor"].str.extract(r"/(Pid\.\w+)/")[0].value_counts()
tahun = meta["tahun"].value_counts().sort_index()
print("jenis perkara\n" + tipe.to_string() + "\ntahun register\n" + tahun.to_string())

jenis perkara
0
Pid.B      134
Pid.Sus     68
Pid.C        6
tahun register
tahun
2025    124
2026     84


Perkara peradilan umum didominasi `Pid.B` (biasa) disusul `Pid.Sus`
(khusus); register terkumpul dari dua tahun sehingga korpus mencakup
rentang perkara 2025-2026.

In [4]:
MONTHS = {"NOPEMBER": "NOVEMBER", "PEBRUARI": "FEBRUARI", "JANUARI": "JANUARI",
          "APRIL": "APRIL", "AGUSTUS": "AGUSTUS", "OKTOBER": "OKTOBER"}
BOILER = re.compile(
    r"^\s*(Mahkamah Agung Republik Indonesia"
    r"|Direktori Putusan Mahkamah Agung Republik Indonesia"
    r"|putusan\.mahkamahagung\.go\.id"
    r"|Disclaimer"
    r"|Kepaniteraan Mahkamah Agung.*"
    r"|pelaksanaan fungsi peradilan.*"
    r"|Dalam hal .*"
    r"|Email : kepaniteraan.*"
    r"|Halaman \d+ dari \d+.*"
    r"|P U T U S A N)\s*$",
    re.MULTILINE,
)


def clean_text(text):
    return BOILER.sub("", text)


def key_of(nomor):
    m = re.match(r"(\d+)/(Pid\.\w+)/(\d{4})", nomor)
    return f"{m.group(1)}-{m.group(2).lower()}-{m.group(3)}" if m else ""


def load_text(key, row):
    cache = TXT_CACHE / f"{key}.txt"
    if cache.exists():
        return cache.read_text(), "pdf"
    match = list(PDF_DIR.glob(f"putusan_{key}_pn_ngw_*.pdf".replace("-", "_")))
    if match:
        raw = "\n".join((pg.extract_text() or "") for pg in PdfReader(match[0]).pages)
        cache.write_text(raw)
        return raw, "pdf"
    parts = [
        f"Putusan Nomor {row['nomor']}",
        "Pengadilan Negeri Ngawi",
        f"Nama lengkap : {row['terdakwa']}",
        f"Hakim Ketua : {row['hakim_ketua']}",
        f"Hakim Anggota : {row['hakim_anggota']}",
        f"Panitera Pengganti : {row['panitera']}",
        f"Penuntut Umum : {row['penuntut_umum']}",
        f"Tanggal dibacakan : {row['tanggal_dibacakan']}",
        "MENGADILI:",
        str(row["catatan_amar"]),
    ]
    return "\n".join(parts), "meta"

In [5]:
sources = {}
for _, row in meta.iterrows():
    _, src = load_text(key_of(row["nomor"]), row)
    sources[src] = sources.get(src, 0) + 1
print(f"teks penuh dari PDF: {sources.get('pdf', 0)} putusan | parsial dari metadata: {sources.get('meta', 0)} putusan")

teks penuh dari PDF: 187 putusan | parsial dari metadata: 21 putusan


Sebagian besar putusan memiliki berkas PDF sehingga teks penuh tersedia.
Sisanya hanya berbekal metadata dan `catatan_amar` yang memuat amar
`MENGADILI`, sehingga entitas tindak pidana dan putusan tetap dapat
dianotasi walau pasal, tuntutan, dan penasihat hukum tidak terbaca.

## B. IOB Annotation

Anotasi memakai dua mesin. *Trie* mencocokkan entitas yang nilainya
tersedia pada metadata: nomor perkara, terdakwa, tanggal, hakim ketua,
hakim anggota, panitera, dan penuntut umum. Ekspresi reguler menangkap
entitas yang hanya muncul di badan teks: tindak pidana, pasal, tuntutan,
putusan, dan penasihat hukum. Token dibersihkan huruf besar semua dan tanda
baca tepi dibuang agar pencocokan konsisten dengan tugas sebelumnya.

### Tag Legend

| Tag | Entitas | Sumber anotasi |
|-----|---------|----------------|
| VERN | nomor perkara | metadata `nomor` |
| DEFN | terdakwa, termasuk varian alias | metadata `terdakwa` |
| CRIA | tindak pidana | regex `MELAKUKAN TINDAK PIDANA ... SEBAGAIMANA` |
| ARTV | pasal | regex `PASAL ...` beserta rujukan undang-undangnya |
| PENA | tuntutan penuntut umum | frasa `PIDANA PENJARA SELAMA ...` sebelum `MENGADILI` |
| PUNI | putusan hakim | frasa yang sama sesudah `MENGADILI` |
| TIMV | tanggal pembacaan | metadata `tanggal_dibacakan` dan `tanggal_musyawarah` |
| JUDP | hakim ketua | metadata `hakim_ketua` |
| JUDG | hakim anggota | metadata `hakim_anggota`, satu span per hakim |
| REGI | panitera | metadata `panitera` |
| PROS | penuntut umum | metadata `penuntut_umum`, cadangan regex dari badan teks |
| ADVO | penasihat hukum | regex `DIDAMPINGI (OLEH)?`, `ADVOKAT`, `PENGACARA`, `PENASIHAT HUKUM` |
| O | token di luar entitas | - |

Awalan `B_` menandai token pertama sebuah entitas dan `I_` lanjutannya,
sedangkan `O` berarti token tidak termasuk entitas mana pun.

In [6]:
def clean_token(tok):
    return re.sub(r"^[^\w]+|[^\w]+$", "", unicodedata.normalize("NFKC", tok).upper())


def split_multi(value):
    return [p.strip() for p in re.split(r"\s*\d+\.\s*|\s+#\s*", value) if p.strip()]


def split_judges(value):
    parts = re.split(r"(?:\bBr\b|\bHakim Anggota\b|\bHakim Ketua\b|,)+", value)
    return [p.strip() for p in parts if p.strip()]


def strip_role(value):
    v = re.sub(r"^(Hakim\s+(Ketua|Anggota|Tunggal)\s*)+", "", value)
    v = re.sub(r"^(Panitera\s+Pengganti\s*:?\s*)+", "", v)
    v = re.sub(r"^(Jaksa\s+Penuntut\s+Umum\s*|Penuntut\s+Umum\s*)+", "", v)
    return v.strip(" :-")


DEGREE = {"SH", "MH", "SHI", "MM", "SE", "ST", "SS", "SI", "SPI", "MKN", "SPN"}
TITLE_RE = re.compile(r"^[A-Z](\.[A-Z])*\.?$")


def norm_date(value):
    m = re.search(r"(\d{1,2})\s+(\w+)\s+(\d{4})", value)
    if not m:
        return value
    return f"{m.group(1)} {MONTHS.get(m.group(2).upper(), m.group(2))} {m.group(3)}"


def name_tokens(value, cut_comma=True):
    v = re.sub(r"\((?:Alm|Almarhum|Almarhumah|Alm\.)\)", "", str(value), flags=re.IGNORECASE)
    if cut_comma and "," in v:
        v = v.split(",")[0]
    toks = [clean_token(t) for t in v.split()]
    return [t for t in toks if t and not TITLE_RE.match(t) and t not in DEGREE]


def variants(tokens):
    out = [list(tokens)]
    interior = list(range(1, len(tokens) - 1))
    if 0 < len(interior) <= 5:
        for r in range(1, len(interior) + 1):
            for combo in itertools.combinations(interior, r):
                out.append([t[0] if i in combo else t for i, t in enumerate(tokens)])
    elif len(interior) > 5:
        out.append([t[0] if i in interior else t for i, t in enumerate(tokens)])
    seen, uniq = set(), []
    for seq in out:
        key = tuple(seq)
        if key not in seen:
            seen.add(key)
            uniq.append(seq)
    return uniq

Gelar pada nama (`S.H.`, `M.H.`) dipotong pada koma pertama dan gelar
akademik dibuang dari token pembanding. Nama panjang dibuat beragam varian
lewat penyingkatan nama tengah, karena di badan putusan tokoh yang sama bisa
ditulis utuh maupun sebagai singkatan.

In [7]:
class TrieNode:
    __slots__ = ("children", "label")

    def __init__(self):
        self.children = {}
        self.label = None


def build_trie(queries):
    root = TrieNode()
    for label, seqs in queries.items():
        for seq in seqs:
            node = root
            for t in seq:
                node = node.children.setdefault(t, TrieNode())
            node.label = label
    return root


def annotate_trie(tokens, root):
    spans = []
    i = 0
    while i < len(tokens):
        node, j, last = root, i, None
        while j < len(tokens):
            nxt = node.children.get(tokens[j])
            if nxt is None:
                break
            node = nxt
            if node.label:
                last = (j, node.label)
            j += 1
        if last:
            spans.append((i, last[0], last[1]))
            i = last[0] + 1
        else:
            i += 1
    return spans


def build_queries(row):
    q = {}

    def add(label, seqs):
        for seq in seqs:
            if seq:
                q.setdefault(label, []).append(seq)

    add("VERN", [name_tokens(row["nomor"], cut_comma=False)])
    for col in ("tanggal_dibacakan", "tanggal_musyawarah"):
        add("TIMV", [name_tokens(norm_date(row[col]), cut_comma=False)])
    for label, col in (("JUDP", "hakim_ketua"), ("REGI", "panitera"),
                       ("PROS", "penuntut_umum")):
        add(label, variants(name_tokens(strip_role(row[col]))))
    for part in split_judges(row["hakim_anggota"]):
        add("JUDG", variants(name_tokens(strip_role(part))))
    raw = re.sub(r"\(Alm(?:arhum(?:ah)?)?\)", "", str(row["terdakwa"]), flags=re.IGNORECASE)
    for person in split_multi(raw):
        for part in re.split(r"\s+(?:Als|Alias|Bin)\s+", person, flags=re.IGNORECASE):
            add("DEFN", variants(name_tokens(part, cut_comma=False)))
    full = name_tokens(raw, cut_comma=False)
    if full:
        add("DEFN", [full])
        add("DEFN", [[("ALIAS" if t == "ALS" else t) for t in full]])
    return q

In [8]:
LAWWORD = (
    "KUHP|KUHAP|KUHPNASIONAL|UNDANG-UNDANG|UNDANG|UU|KITAB|HUKUM|PIDANA|PERATURAN|REPUBLIK"
    "|INDONESIA|NOMOR|TAHUN|TENTANG|AYAT|NASIONAL|PERLINDUNGAN|ANAK|NARKOTIKA|KESEHATAN"
    "|PEMERINTAH|PERUBAHAN|PENETAPAN|ATAS|KEKERASAN|SEKSUAL|INFORMASI|TRANSAKSI|ELEKTRONIK|ITE"
    "|PERKAWINAN|PENYELESAIAN|ANGKUTAN|JALAN|SENJATA|PERDAGANGAN|ORANG|PENGHAPUSAN|PAJAK|KHUSUS|URUSAN"
)
ARTV = re.compile(rf"PASAL \d+(?:-\d+)?[A-Z]*(?: (?:{LAWWORD}|\d+))*")
CRIA = re.compile(r"MELAKUKAN TINDAK PIDANA (.{2,120}?) SEBAGAIMANA")
AMAR = re.compile(r"MENGADILI|M E N G A D I L I")
PHRASE = re.compile(r"PIDANA PENJARA SELAMA \d+ (?:[A-Z]+ )?(?:TAHUN|BULAN|HARI)")

ADVO_STOP = {
    "HUKUM", "DKK", "DAN", "ADVOKAT", "YANG", "BERKANTOR", "BERALAMAT", "ALAMAT", "DI",
    "JALAN", "PADA", "TERSEBUT", "PENGADILAN", "PBH", "PERADI", "NGAWI", "YAITU", "BERNAMA",
    "YAKNI", "DARI", "KANTOR", "TIDAK", "DIDAMPINGI", "OLEH", "SDR", "SAUDARA", "ANAK",
    "ISTRI", "POS", "BAKSO", "LIAR", "SURAT", "PENETAPAN", "KETUA", "NOMOR", "TANGGAL",
    "IZIN", "KEPADA", "PENASIHAT", "PENASEHAT", "PENASIAT", "NEGERI", "DENGAN", "SECARA",
    "LISAN", "MEMBACA", "SETELAH", "UP", "MENGADILI", "MENURUT", "AGAMA", "SIDANG",
    "TEMPAT", "LAHIR", "UMUR", "SUSUNAN", "PERSIDANGAN", "NYA", "DIPERSIDANGAN",
    "BERSUMPAH", "SELAKU", "TERDAKWA", "MENYATAKAN", "TETAP", "MENGERTI", "SUDAH", "AKAN",
    "MENOLAK", "MENUNTUT", "MENGANGGAP", "TERHADAP", "BERSAMA", "MEMBERI", "KEBEBASAN",
    "PENGACARA", "LBH", "POSBAKUM", "KABUPATEN", "PROVINSI", "KOTA", "KECAMATAN", "DESA",
    "KELURAHAN", "IDENTITAS", "BERDASARKAN", "SENDIRI", "MENGENAI", "KANWIL",
    "HAKIM", "ANGGOTA", "MAJELIS", "PANITERA", "PENGGANTI", "PENUNTUT", "JAKSA", "SAKSI",
    "RUMAH", "INDONESIA", "MEMBERIKAN", "TELAH", "BAHWA", "PROSES", "PENYELIDIKAN",
    "HADIR", "TAPI", "ATAU", "AAN", "DITUNJUK", "BUKAN", "ATAS", "KEMAUAN",
    "PENGURUS", "PONDOK", "PESANTREN", "SETIAP", "HARI", "KONSULTAN", "TEDAKWA",
    "PERKENANKANLAH", "KAMI", "UNTUK", "KEPENTINGAN", "BELIAU", "FOTO", "PARAF",
    "DIKARENAKAN", "BELUM", "ADANYA", "SEDANGKAN", "SAAT", "DILAKUKAN", "PEMERIKSAAN",
    "HAKIM-HAKIM", "SDRI", "HUKUM/ADVOKAT",
}
ADVO_LEAD = {"YANG", "BERNAMA", "YAITU", "YAKNI", "SDR", "SAUDARA"}
ADVO_SKIP = ADVO_LEAD | {
    "PARA", "DAN", "SERTA", "OLEH", "DKK", "HUKUM", "ADVOKAT", "PENGACARA",
    "PENASIHAT", "PENASEHAT", "PENASIAT", "DIDAMPINGI",
    "HUKUMNYA", "ADVOKATNYA", "PENGACARANYA", "PENAS", "IHAT", "SDRI",
}
PENASI = {"PENASIHAT", "PENASEHAT", "PENASIAT"}
ADVO_WORD = {"ADVOKAT", "PENGACARA", "DIDAMPINGI"}
PROS_STOP = {
    "UMUM", "KEJAKSAAN", "TINGGI", "NEGERI", "NGAWI", "PADA", "YANG", "DAN", "SERTA",
    "MENUNTUT", "PENUNTUTAN", "TUNTUTAN", "SDR", "SAUDARA", "NAMA", "NOMOR", "TANGGAL",
    "SURAT", "IZIN", "PENETAPAN", "KEPADA", "DENGAN", "OLEH", "SELAKU", "MEMBACA", "PIDANA",
    "MENGADILI", "MENYATAKAN", "DIDAMPINGI", "TERDAKWA", "HUKUM", "ADVOKAT", "PENGACARA",
    "PENASIHAT", "PENASEHAT", "PENASIAT", "BERKANTOR", "BERALAMAT", "JAKSA", "KEPALA",
    "WILAYAH", "PERKARA", "PUTUSAN", "SELAMA", "DALAM", "PUSAT", "CABANG",
    "HAKIM", "ANGGOTA", "MAJELIS", "PANITERA", "PENGGANTI", "SAKSI",
    "SEJAK", "MENGAJUKAN", "SAKSI-SAKSI", "SEBAGAI", "BARANG", "BUKTI", "BERDASARKAN",
    "DIDAKWA", "TERHADAP", "PERMOHONAN", "MENIMBANG", "BAHWA", "PEMBELAAN", "BENAR",
    "AHLI", "TERPENUHI", "MENGHADAPKAN", "DARI", "PERTIMBANGAN", "TINDAK", "SEHINGGA",
    "KEPADANYA", "LISAN", "SECARA", "NOTA", "AKAN", "MELAMPAUI", "KADAR", "KESALAHAN",
    "PERIHAL", "MAKA", "PULA", "DIBACAKAN", "DITUNTUT", "MELEPASKAN", "SETIDAK-TIDAKNYA",
    "ORANG", "BERNAMA", "TETAP", "DIDALAM", "DIPEROLEH", "DIPANDANG", "DIMINTAKAN",
    "SELAIN", "DEMIKIAN", "LAGI", "INILAH", "MEMBUKTIKAN", "UNTUK", "MENJATUHKAN",
    "TERSEBUT", "MASIH", "MENDASARKAN", "MENGHADIRKAN", "KETERANGAN", "DAPAT",
    "DIPERGUNAKAN", "PARA", "SEORANG", "TIDAK",
}


def name_run(tokens, start, limit, stop):
    j = start
    while j < len(tokens) and j - start < limit:
        t = tokens[j]
        if t.isdigit() or len(t) < 2 or ("." in t and len(t) <= 8) or t in stop:
            break
        j += 1
    return j


def advo_spans(tokens):
    out = []
    i = 0
    while i < len(tokens):
        if tokens[i] in PENASI and i + 1 < len(tokens) and tokens[i + 1] == "HUKUM":
            end = i + 2
        elif tokens[i] in ADVO_WORD:
            end = i + 1
            if tokens[i] == "DIDAMPINGI" and end < len(tokens) and tokens[end] == "OLEH":
                end += 1
        else:
            i += 1
            continue
        j = end
        guard = 0
        while j < len(tokens) and tokens[j] in ADVO_SKIP and guard < 8:
            j += 1
            guard += 1
        start = j
        j = name_run(tokens, start, 5, ADVO_STOP)
        if j > start:
            out.append((start, j - 1, "ADVO"))
            i = j
        else:
            i = end
    return out


def pros_spans(tokens):
    out = []
    i = 0
    while i < len(tokens):
        if tokens[i] == "PENUNTUT" and i + 1 < len(tokens) and tokens[i + 1] == "UMUM":
            end = i + 2
        elif tokens[i] == "JAKSA":
            end = i + 1
            if end + 1 < len(tokens) and tokens[end] == "PENUNTUT" and tokens[end + 1] == "UMUM":
                end += 2
            elif end < len(tokens) and tokens[end] in ("TINGGI", "NEGERI", "KEPALA"):
                end += 1
        else:
            i += 1
            continue
        j = end
        while j < len(tokens) and tokens[j] in ADVO_LEAD:
            j += 1
        start = j
        j = name_run(tokens, start, 4, PROS_STOP)
        if j > start and all(len(t) >= 3 for t in tokens[start:j]):
            out.append((start, j - 1, "PROS"))
            i = j
        else:
            i = end
    return out


def char_to_token(offsets, start, end):
    first = next((i for i, (s, _) in enumerate(offsets) if s >= start), len(offsets))
    last = first - 1
    for i, (_, e) in enumerate(offsets):
        if e <= end:
            last = i
        else:
            break
    return first, last


def regex_spans(tokens, joined, offsets):
    out = []

    def add(label, cs, ce):
        a, b = char_to_token(offsets, cs, ce)
        if a <= b:
            out.append((a, b, label))

    for m in CRIA.finditer(joined):
        add("CRIA", m.start(1), m.end(1))
    for m in ARTV.finditer(joined):
        add("ARTV", m.start(), m.end())
    amar = list(AMAR.finditer(joined))
    cut = len(tokens)
    if amar:
        cut = char_to_token(offsets, amar[-1].end(), amar[-1].end())[0]
    for m in PHRASE.finditer(joined):
        a, b = char_to_token(offsets, m.start(), m.end())
        if a <= b:
            out.append((a, b, "PUNI" if a >= cut else "PENA"))
    out.extend(advo_spans(tokens))
    return out


def fallback_spans(tokens, present):
    out = []
    if "VERN" not in present:
        for i, t in enumerate(tokens[:60]):
            if t == "NOMOR" and i + 1 < len(tokens) and re.search(r"\d|X", tokens[i + 1]):
                end = i + 1
                if end + 1 < len(tokens) and tokens[end + 1] in ("NGW", "NGAWI"):
                    end += 1
                out.append((i + 1, end, "VERN"))
                break
    if "DEFN" not in present:
        defn = []
        i = 0
        while i < len(tokens):
            if re.fullmatch(r"X{2,}", tokens[i]):
                j = i
                while j + 1 < len(tokens) and re.fullmatch(r"X{2,}", tokens[j + 1]):
                    j += 1
                defn.append((i, j, "DEFN"))
                i = j + 1
            else:
                i += 1
        if not defn:
            for i in range(len(tokens) - 2):
                if tokens[i] == "NAMA" and tokens[i + 1] == "LENGKAP":
                    defn.append((i + 2, i + 2, "DEFN"))
                    break
        out += defn
    if "PROS" not in present:
        out += pros_spans(tokens)
    return out


Frasa tuntutan dan putusan dipisah pada posisi `MENGADILI`: `PIDANA
PENJARA SELAMA ...` sebelum amar dibaca sebagai tuntutan penuntut umum,
sesudahnya sebagai putusan hakim; bentuk vonis yang tidak memakai pola itu
(misalnya pidana denda) tidak ikut terlabeli. Pencarian nama dihentikan oleh
kata peran seperti `HAKIM` atau `PANITERA` supaya rentang nama tidak menyerobot
blok peran berikutnya. Tiga cadangan melengkapi trie dan regex bila sebuah
entitas belum menemukan apa pun: nomor perkara lewat kata `NOMOR`, nama
terdakwa lewat token `X` berulang atau `NAMA LENGKAP`, dan penuntut umum lewat
penanda dari badan teks.

In [9]:
def annotate_doc(text, row):
    text = clean_text(text)
    tokens, frag_of, offsets = [], [], []
    cursor = 0
    for frag_no, chunk in enumerate(text.split(";")):
        for raw in chunk.split():
            tok = clean_token(raw)
            if not tok:
                continue
            tokens.append(tok)
            frag_of.append(frag_no)
            if cursor:
                cursor += 1
            offsets.append((cursor, cursor + len(tok)))
            cursor += len(tok)
    joined = " ".join(tokens)

    spans = list(annotate_trie(tokens, build_trie(build_queries(row))))
    spans += regex_spans(tokens, joined, offsets)
    present = {label for _, _, label in spans}
    spans += fallback_spans(tokens, present)

    spans.sort(key=lambda s: (s[0], -(s[1] - s[0])))
    tags = ["O"] * len(tokens)
    for a, b, label in spans:
        if any(tags[k] != "O" for k in range(a, b + 1)):
            continue
        tags[a] = "B_" + label
        for k in range(a + 1, b + 1):
            tags[k] = "I_" + label
    for k in range(1, len(tags)):
        if frag_of[k] != frag_of[k - 1] and tags[k].startswith("I_"):
            tags[k] = "B_" + tags[k][2:]
    return [(frag, tok, tag) for tok, frag, tag in zip(tokens, frag_of, tags)]


ds = pd.DataFrame(
    [(row["nomor"], frag, tok, tag)
     for _, row in meta.iterrows()
     for frag, tok, tag in annotate_doc(load_text(key_of(row["nomor"]), row)[0], row)],
    columns=["doc", "sentence", "token", "tag"])
n_frag = ds.groupby("doc")["sentence"].nunique().sum()
print(f"{len(ds)} baris | {ds['doc'].nunique()} dokumen | {n_frag} fragmen | kolom {list(ds.columns)}")

1223429 baris | 208 dokumen | 29382 fragmen | kolom ['doc', 'sentence', 'token', 'tag']


Setiap dokumen dipecah menjadi fragmen pada titik koma, lalu seluruh token
dan labelnya dirangkai menjadi tabel tunggal. Kolom `doc` memuat nomor
perkara, `sentence` nomor fragmen, `token` potongan teks, dan `tag` label
IOB-nya. Bila sebuah entitas terpotong batas fragmen, penandanya dimulai
ulang dengan `B_` pada fragmen berikutnya agar urutan IOB tetap sah di
setiap fragmen.

In [10]:
LABELS = ["VERN", "DEFN", "CRIA", "ARTV", "PENA", "PUNI", "TIMV",
          "JUDP", "JUDG", "REGI", "PROS", "ADVO"]
has = ds.assign(core=ds["tag"].str.slice(2)).groupby(["doc", "core"]).size().unstack(fill_value=0) > 0
rows = [(lab, int(has[lab].sum()) if lab in has else 0, len(has)) for lab in LABELS]
print(pd.DataFrame(rows, columns=["entitas", "dokumen berlabel", "total dokumen"]).to_string(index=False))

entitas  dokumen berlabel  total dokumen
   VERN               208            208
   DEFN               208            208
   CRIA               192            208
   ARTV               187            208
   PENA               154            208
   PUNI               182            208
   TIMV               207            208
   JUDP               190            208
   JUDG               208            208
   REGI               206            208
   PROS               202            208
   ADVO                63            208


Tabel cakupan mengukur berapa dokumen yang benar-benar memiliki tiap
entitas. Nomor perkara, terdakwa, pasal, tanggal, hakim anggota, dan vonis
hampir selalu hadir karena tersedia pada metadata atau pola amar. Entitas
tuntutan dan penasihat hukum lebih jarang karena hanya muncul pada berkas
yang teks penuhnya terekstrak.

In [11]:
dist = ds["tag"].value_counts().rename_axis("tag").reset_index(name="count")
dist["pct"] = (dist["count"] / len(ds) * 100).round(2)
print(dist.to_string(index=False))

   tag   count   pct
     O 1184604 96.83
I_DEFN   10320  0.84
I_ARTV    8558  0.70
B_DEFN    6347  0.52
I_CRIA    2310  0.19
B_ARTV    2213  0.18
I_JUDG    1449  0.12
I_PUNI     931  0.08
I_JUDP     818  0.07
I_PENA     794  0.06
B_JUDG     759  0.06
B_VERN     591  0.05
I_VERN     589  0.05
B_CRIA     447  0.04
I_TIMV     446  0.04
B_REGI     394  0.03
B_JUDP     355  0.03
I_REGI     326  0.03
I_PROS     246  0.02
B_TIMV     223  0.02
B_PROS     221  0.02
B_PUNI     187  0.02
B_PENA     159  0.01
I_ADVO      77  0.01
B_ADVO      65  0.01


Distribusi label sangat timpang: kelas `O` mendominasi seperti lazimnya
tugas NER, sedangkan `DEFN` dan `ARTV` menjadi entitas terbanyak karena
nama terdakwa muncul berulang dan pasal disebut berkali-kali di sepanjang
putusan.

## C. Dataset Checks

Pemeriksaan memastikan urutan IOB konsisten, yaitu setiap token `I_` selalu
didahului `B_` atau `I_` dengan jenis entitas yang sama. Contoh anotasi
ditampilkan dengan identitas orang dimasker menjadi inisial dan nomor
perkara menjadi `N°`, agar halaman publik ini tidak memuat data pribadi.

### C.2 Precision audit (ADVO & PROS)

Pemeriksaan berbasis output aturan mentah menunjukkan sebagian besar span setelah penanda peran adalah kata fungsi, bukan nama. Stoplist hasil audit digunakan agar hanya nama sungguhan yang berlabel `B_ADVO`/`I_ADVO` maupun `B_PROS`/`I_PROS`.


In [12]:
bad = 0
for _, grp in ds.groupby(["doc", "sentence"], sort=False):
    prev = ""
    for tag in grp["tag"]:
        if tag.startswith("I_") and prev != "B_" + tag[2:] and prev != tag:
            bad += 1
        prev = tag
print(f"pelanggaran urutan IOB: {bad}")

pelanggaran urutan IOB: 0


Tidak ditemukan pelanggaran urutan IOB, sehingga dataset dapat langsung
dipakai oleh model *sequence labeling* seperti CRF pada tugas berikutnya.

In [13]:
def pseudonym(tag, tokens):
    core = tag.split("_")[-1]
    if core == "VERN":
        return "N°"
    if core in ("DEFN", "PROS", "JUDP", "JUDG", "REGI", "ADVO"):
        return " ".join(w[0] + "." if w.isalpha() else w for w in tokens)
    return " ".join(tokens)


examples, seen = [], set()
for _, grp in ds.groupby(["doc", "sentence"], sort=False):
    seq = list(zip(grp["tag"], grp["token"]))
    for tag, chunk in itertools.groupby(seq, key=lambda x: x[0]):
        if tag == "O":
            continue
        toks = tuple(t for _, t in chunk)
        if (tag, toks) in seen:
            continue
        seen.add((tag, toks))
        examples.append((tag, pseudonym(tag, toks)))
    if len(examples) >= 400:
        break
ex = pd.DataFrame(examples, columns=["label", "entitas"]).sample(frac=1, random_state=42).head(16)
print(ex.to_string(index=False))

 label                                                                            entitas
I_ARTV                                      435 UNDANG-UNDANG NOMOR 17 TAHUN 2023 TENTANG
I_DEFN                                                                        D. S. B. T.
I_ARTV                                                                                 20
I_PROS                                                                                 H.
I_ARTV                                                                         138 AYAT 2
I_DEFN                                                                              A. R.
B_VERN                                                                                 N°
I_ARTV                                                                             435 UU
B_DEFN                                                                                 K.
I_CRIA SENGAJA MEMBUJUK ANAK MELAKUKAN PERSETUBUHAN DENGANNYA SEBAGAI PERBUATAN BERLANJUT
I_JUDG    

Contoh acak memperlihatkan tiap label menangkap potongan yang tepat: nomor
perkara sebagai `N°`, nama sebagai inisial, pasal berupa frasa `Pasal ...`,
serta tindak pidana, tuntutan, dan vonis dalam bentuk frasa lengkap.

### C.1 Corpus Verification

Selain urutan IOB, dataset diverifikasi keterlacakannya: setiap berkas yang
dianotasi harus berasal dari daftar unduhan sendiri dan tidak boleh memuat
putusan pengadilan lain. Pemeriksaan pertama membandingkan daftar unduhan
dengan berkas PDF yang tersimpan, termasuk berkas ganda.

In [14]:
unduh = pd.read_csv(DATA / "dataset" / "unduh" / "daftar-unduhan.csv", dtype=str)
berkas_pdf = sorted(PDF_DIR.glob("*.pdf"))
grup = {}
for f in berkas_pdf:
    grup.setdefault(re.sub(r"(_\d{14})(-\d+)?\.pdf$", "", f.name), []).append(f)
ganda = {k: v for k, v in grup.items() if len(v) > 1}


def sidik(path):
    teks = "\n".join((pg.extract_text() or "") for pg in PdfReader(str(path)).pages)
    return hashlib.md5(teks.encode()).hexdigest()


identik = sum(1 for v in ganda.values() if len({sidik(f) for f in v}) == 1)
pengadilan = ", ".join(sorted(unduh["nomor"].str.split("/").str[-1].unique()))
print(f"tautan unduhan: {unduh['tipe'].value_counts().to_dict()} | pengadilan: {pengadilan}")
print(f"berkas PDF: {len(berkas_pdf)} | putusan unik: {len(grup)} | berkas ganda berisi identik: {identik}/{len(ganda)}")

tautan unduhan: {'pdf': 187, 'detail': 21} | pengadilan: PN Ngw
berkas PDF: 195 | putusan unik: 187 | berkas ganda berisi identik: 8/8


Daftar unduhan menautkan seluruh 208 putusan dan semuanya milik satu
pengadilan, yaitu PN Ngawi. Jumlah berkas PDF lebih banyak daripada putusan
unik karena sebagian terunduh berulang; sidik isi membuktikan berkas ganda
itu identik, sehingga tidak ada dokumen tambahan di luar korpus.

In [15]:
berkas_teks = sorted(TXT_CACHE.glob("*.txt"))
teks = {f.stem: re.sub(r"\s+", " ", f.read_text().upper()) for f in berkas_teks}
pidana_umum = int(meta["nomor"].str.contains(r"/Pid\.(?:B|Sus|C)/").sum())
ngw = int(unduh["nomor"].str.endswith("PN Ngw").sum())
lain = int((~unduh["nomor"].str.endswith("PN Ngw")).sum())
merujuk = sum(1 for t in teks.values() if "PENGADILAN NEGERI NGAWI" in t or "PN NGW" in t)
resmi = sum(1 for t in teks.values() if "PUTUSAN.MAHKAMAHAGUNG.GO.ID" in t)
tertutup = sum(1 for t in teks.values() if re.search(r"P U T U S A N NOMOR X", t))
ponorogo = sum(1 for t in teks.values() if "PENGADILAN NEGERI PONOROGO" in t)

rows = [
    ("putusan teranotasi", len(meta), len(meta)),
    ("nomor berakhiran /PN Ngw", ngw, len(unduh)),
    ("kategori pidana umum", pidana_umum, len(meta)),
    ("berkas dari domain resmi", resmi, len(berkas_teks)),
    ("teks merujuk Pengadilan Negeri Ngawi", merujuk, len(berkas_teks)),
    ("putusan dari pengadilan lain", lain, len(meta)),
]
print(pd.DataFrame(rows, columns=["pemeriksaan", "sesuai", "total"]).to_string(index=False))
print(f"nomor perkara ter-redaksi sebagian atau penuh: {tertutup} berkas | menyebut pengadilan lain: {ponorogo} berkas")

                         pemeriksaan  sesuai  total
                  putusan teranotasi     208    208
            nomor berakhiran /PN Ngw     208    208
                kategori pidana umum     208    208
            berkas dari domain resmi     187    187
teks merujuk Pengadilan Negeri Ngawi     187    187
        putusan dari pengadilan lain       0    208
nomor perkara ter-redaksi sebagian atau penuh: 17 berkas | menyebut pengadilan lain: 1 berkas


Seluruh nomor perkara pada daftar unduhan berakhiran `/PN Ngw` dan seluruh
berkas berasal dari domain resmi, sehingga tidak ada putusan pengadilan
lain yang ikut terpakai. Dua hal dicatat sebagai keterbatasan. Sebagian
berkas pidana khusus menampilkan nomor perkara ter-redaksi sebagian atau
penuh, sehingga identitasnya dipastikan dari metadata dan daftar unduhan,
bukan dari badan teks. Selain itu ada berkas yang menyebut Pengadilan
Negeri Ponorogo secara insidental di dalam narasi, sedangkan dokumennya
sendiri tetap putusan PN Ngawi.

## D. Dataset Export

Dataset disimpan lokal pada `data/IE/dataset/` bersama satu berkas catatan
yang memuat nama pengadilan, waktu scraping, jumlah putusan, dan nama
pemeriksa. Sesuai etika pengumpulan, isi baris memuat nama orang sehingga
hanya disimpan lokal dan tidak dipublikasikan; halaman ini menampilkan
agregat saja.

In [16]:
ds.to_csv(OUT_CSV, index=False)
PEMERIKSA = [n.strip() for n in (OUT_TXT.parent / "pemeriksa.txt").read_text().splitlines() if n.strip()]
catatan = "\n".join([
    "CATATAN DATASET ANOTASI IE",
    "",
    "Pengadilan Negeri : Pengadilan Negeri Ngawi (PN Ngawi)",
    "Waktu scraping    : 25 September 2026",
    "Jumlah putusan    : 208 putusan",
    "",
    "Pemeriksa dataset :",
    f"  1. {PEMERIKSA[0]}",
    f"  2. {PEMERIKSA[1]}",
    "",
    "Skema entitas (IOB): VERN, DEFN, CRIA, ARTV, PENA, PUNI, TIMV,",
    "JUDP, JUDG, REGI, PROS, ADVO.",
    "",
    "Arti tag:",
    "  VERN nomor perkara     DEFN terdakwa          CRIA tindak pidana",
    "  ARTV pasal             PENA tuntutan umum     PUNI putusan hakim",
    "  TIMV tanggal           JUDP hakim ketua       JUDG hakim anggota",
    "  REGI panitera          PROS penuntut umum     ADVO penasihat hukum",
    "  O    token di luar entitas",
])
OUT_TXT.write_text(catatan)
print(f"{OUT_CSV.name}: {len(ds)} baris, {ds['doc'].nunique()} dokumen\n{OUT_TXT.name}: catatan pengumpulan")

PN-NGW-230411100197.csv: 1223429 baris, 208 dokumen
PN-NGW-230411100197.txt: catatan pengumpulan


Keluaran final berisi 208 dokumen dengan format `doc, sentence, token,
tag`, disertai berkas catatan terpisah sesuai permintaan tugas. Nama kedua
pemeriksa dibaca dari berkas lokal, bukan ditulis pada halaman ini.

## Conclusion

Dataset anotasi IOB berhasil dibangun dari 208 putusan pidana umum PN
Ngawi. Teks diperoleh dari 187 berkas PDF dan 21 dokumen dianotasi dari
metadata serta `catatan_amar`. Dua belas entitas dilabeli memakai *trie*
untuk nilai metadata dan ekspresi reguler untuk entitas badan teks, dengan
cadangan untuk nilai ter-redaksi. Hasil pemeriksaan menunjukkan nol
pelanggaran urutan IOB, berkas catatan memuat nama dua anggota pemeriksa
dari kelompok yang sama, dan seluruh data level baris disimpan lokal pada
`data/IE/dataset/PN-NGW-230411100197.csv`.